# GALVAT Trial 9: Balanced Roboflow Training, Evaluation, and Export

This notebook downloads Roboflow dataset version 3 directly, verifies its complete
44-class GALVAT mapping and split integrity, trains YOLO26s on an A100 without rewriting
or filtering any image, and selects the **checkpoint, NMS IoU, and per-class confidence
thresholds using validation GALVAT weighted score**. Test is evaluated once with frozen
validation-selected settings.

Primary output: full-ontology GALVAT weighted score and a 44-row per-class report.
Standard AP, background-RoI false positives, spoken-output burden, PyTorch weights,
ONNX, thresholds, charts, and audit manifests are also saved to Drive.

The Roboflow export is used in place. Images are not resized, auto-oriented, re-encoded,
quality-filtered, deduplicated, or relabeled. Ultralytics performs its normal runtime
letterboxing. Exact cross-split duplicates stop the run so leakage is fixed at the source.

**Overnight use:** choose an A100 runtime and run Cell 1 once so Drive authorization
completes, then use **Runtime -> Run all**. The Drive run folder has a fixed name.
Training resumes from `last.pt`, completed validation pairs are cached, and the frozen
test result is locked. If Colab disconnects, reconnect and run all cells again. The
Roboflow API key is requested securely with `getpass` and is never written to Drive.

In [ ]:
# Cell 1: Fixed environment, A100 guard, Drive, and run configuration
!pip -q install ultralytics==8.4.102 roboflow onnx onnxruntime pyyaml pandas seaborn tqdm pillow

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Mapping, Sequence
from getpass import getpass
import datetime, gc, hashlib, json, math, os, random, shutil, zipfile

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

import torch
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
GPU_NAME = torch.cuda.get_device_name(0)
print('Ultralytics:', ultralytics.__version__)
print('Torch:', torch.__version__)
print('GPU:', GPU_NAME)
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
assert 'A100' in GPU_NAME.upper(), f'This run is configured for A100, found: {GPU_NAME}'

ROBOFLOW_WORKSPACE = 'wazir-ali-haideri'
ROBOFLOW_PROJECT = '15-priority-classes'
ROBOFLOW_VERSION = 3
ROBOFLOW_FORMAT = 'yolo26'
DOWNLOAD_ROOT = Path('/content/galvat_trial9_roboflow_v3')
DRIVE_RUNS = Path('/content/drive/MyDrive/galvat_trial9_training_runs')

RUN_STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
# Stable name is intentional: rerunning all cells resumes the same Drive checkpoints.
RUN_NAME = 'galvat_trial9_yolo26s_640_balanced_roboflow_v3_2026-08-08_v1'
RUN_ROOT = DRIVE_RUNS / RUN_NAME
AUDIT_DIR = RUN_ROOT / 'dataset_audit'
SELECTION_DIR = RUN_ROOT / 'validation_selection'
TEST_DIR = RUN_ROOT / 'test_frozen'
CHART_DIR = RUN_ROOT / 'charts'
EXPORT_DIR = RUN_ROOT / 'export'
for path in [RUN_ROOT, AUDIT_DIR, SELECTION_DIR, TEST_DIR, CHART_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_CLASSES = 44
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

MODEL_INIT = 'yolo26s.pt'
IMGSZ = 640
EPOCHS = 120
SAVE_PERIOD = 10
MATCH_IOU = 0.50
NMS_IOU_VALUES = [0.45, 0.55, 0.65, 0.75]
CONF_VALUES = [round(x, 2) for x in np.arange(0.05, 0.96, 0.05)]
PRED_MIN_CONF = 0.025
PRED_BATCH = 8
MAX_DET = 300

print('Roboflow source:', f'{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}')
print('Download root:', DOWNLOAD_ROOT)
print('Run root:', RUN_ROOT)

In [ ]:
# Cell 2: Download Roboflow version 3 and establish a strict GALVAT class mapping
GALVAT_NAMES = [
    'tupperware', 'lemon', 'jug', 'canned_good', 'fork', 'knife', 'spoon',
    'drinking_glass', 'wine_glass', 'coffee_mug', 'salt_pepper_shaker',
    'measuring_cup', 'medicine_bottle', 'pot', 'pan', 'cellphone', 'wallet',
    'purse', 'cane', 'keys', 'toilet', 'urinal', 'paper_towel_dispenser',
    'soap_dispenser', 'trash_can', 'light_switch', 'electrical_outlet',
    'railing', 'tv_remote', 'writing_device', 'eyeglasses', 'hearing_aid',
    'toothbrush', 'toothpaste', 'deodorant', 'razor', 'stairs', 'exit_sign',
    'bathroom_sign', 'thermostat', 'door', 'window', 'chair', 'table',
]
assert len(GALVAT_NAMES) == EXPECTED_CLASSES

existing_yamls = sorted(DOWNLOAD_ROOT.rglob('data.yaml')) if DOWNLOAD_ROOT.exists() else []
if existing_yamls:
    print('Reusing downloaded Roboflow dataset:', DOWNLOAD_ROOT)
else:
    if DOWNLOAD_ROOT.exists():
        shutil.rmtree(DOWNLOAD_ROOT)
    api_key = getpass('Roboflow API key: ')
    assert api_key.strip(), 'Roboflow API key is required.'
    rf = Roboflow(api_key=api_key.strip())
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    version.download(model_format=ROBOFLOW_FORMAT, location=str(DOWNLOAD_ROOT))
    del api_key, rf, project, version

yaml_candidates = sorted(DOWNLOAD_ROOT.rglob('data.yaml'), key=lambda p: (len(p.parts), str(p)))
assert yaml_candidates, f'No data.yaml found under {DOWNLOAD_ROOT}'
SOURCE_DATA_YAML = yaml_candidates[0]
DATA_ROOT = SOURCE_DATA_YAML.parent

with SOURCE_DATA_YAML.open() as handle:
    data_cfg = yaml.safe_load(handle)
source_names_obj = data_cfg['names']
if isinstance(source_names_obj, dict):
    SOURCE_NAMES = [
        source_names_obj[i] if i in source_names_obj else source_names_obj[str(i)]
        for i in range(len(source_names_obj))
    ]
else:
    SOURCE_NAMES = list(source_names_obj)

SOURCE_NAMES = [str(name) for name in SOURCE_NAMES]

# Roboflow version 3 has one known naming difference from GALVAT.
SOURCE_NAME_OVERRIDES = {
    'cell_phone': 'cellphone',
}
NAMES = [SOURCE_NAME_OVERRIDES.get(name, name) for name in SOURCE_NAMES]

unexpected_source_names = sorted(set(NAMES) - set(GALVAT_NAMES))
missing_source_names = sorted(set(GALVAT_NAMES) - set(NAMES))
assert not unexpected_source_names, (
    f'Roboflow contains unmapped class names: {unexpected_source_names}'
)
assert not missing_source_names, (
    f'Roboflow is missing GALVAT classes: {missing_source_names}'
)
assert len(NAMES) == EXPECTED_CLASSES, f'Expected 44 classes, found {len(NAMES)}: {SOURCE_NAMES}'
assert len(set(NAMES)) == EXPECTED_CLASSES, f'Roboflow classes collide after mapping: {NAMES}'
name_to_id = {name: i for i, name in enumerate(NAMES)}

split_dir_names = {'train': 'train', 'valid': 'valid', 'test': 'test'}
for split, directory_name in split_dir_names.items():
    assert (DATA_ROOT / directory_name / 'images').is_dir(), f'Missing {split} images directory'
    assert (DATA_ROOT / directory_name / 'labels').is_dir(), f'Missing {split} labels directory'

# Point to the untouched export while embedding canonical GALVAT names in the model.
runtime_data_cfg = {
    'path': str(DATA_ROOT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(NAMES),
    'names': {index: name for index, name in enumerate(NAMES)},
}
DATA_YAML = RUN_ROOT / 'trial9_direct_data.yaml'
if DATA_YAML.exists():
    assert yaml.safe_load(DATA_YAML.read_text()) == runtime_data_cfg, (
        'Existing Trial 9 runtime data.yaml does not match this export and mapping. '
        'Increment RUN_NAME before changing the dataset.'
    )
else:
    DATA_YAML.write_text(yaml.safe_dump(runtime_data_cfg, sort_keys=False))

source_mapping_df = pd.DataFrame({
    'source_id': range(len(SOURCE_NAMES)),
    'roboflow_class': SOURCE_NAMES,
    'galvat_class': NAMES,
})
source_mapping_df.to_csv(AUDIT_DIR / 'roboflow_to_galvat_class_mapping.csv', index=False)
display(source_mapping_df)
print('SOURCE_DATA_YAML:', SOURCE_DATA_YAML)
print('RUNTIME_DATA_YAML:', DATA_YAML)
print('DATA_ROOT:', DATA_ROOT)
print('Strictly mapped classes:', len(NAMES))

In [ ]:
# Cell 3: Read-only dataset, label, balance, and exact-leakage audit
SPLITS = ['train', 'valid', 'test']

def label_path_for_image(image_path):
    return image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'

def split_images(split):
    image_dir = DATA_ROOT / split / 'images'
    paths = sorted(
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )
    assert len({path.name for path in paths}) == len(paths), f'Duplicate basenames in {split}'
    return paths

def parse_label_file(label_path, split='', image_name=''):
    rows, issues = [], []
    if not label_path.exists():
        return rows, [{
            'split': split, 'file': image_name, 'line': 0,
            'reason': 'missing_label_file', 'raw': '',
        }]
    for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split()
        reason = None
        parsed_row = None
        try:
            class_value = float(parts[0])
            class_id = int(class_value)
            coordinates = [float(value) for value in parts[1:]]
        except Exception:
            reason = 'parse_error'
        else:
            if class_value != class_id:
                reason = 'noninteger_class_id'
            elif class_id < 0 or class_id >= len(NAMES):
                reason = 'bad_class_id'
            elif not all(math.isfinite(value) for value in coordinates):
                reason = 'nonfinite_coordinate'
            elif len(coordinates) == 4:
                x, y, w, h = coordinates
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    reason = 'bad_box_coordinates'
                else:
                    parsed_row = (class_id, x, y, w, h)
            elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
                if not all(0 <= value <= 1 for value in coordinates):
                    reason = 'bad_polygon_coordinates'
                else:
                    xs = coordinates[0::2]
                    ys = coordinates[1::2]
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)
                    w, h = x_max - x_min, y_max - y_min
                    if w <= 0 or h <= 0:
                        reason = 'degenerate_polygon'
                    else:
                        x = (x_min + x_max) / 2
                        y = (y_min + y_max) / 2
                        parsed_row = (class_id, x, y, w, h)
            else:
                reason = 'bad_column_count'
        if reason:
            issues.append({
                'split': split, 'file': image_name, 'line': line_number,
                'reason': reason, 'raw': line,
            })
        else:
            rows.append(parsed_row)
    return rows, issues

def sha1_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha1()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def xywh_to_xyxy(row):
    _, x, y, w, h = row
    return np.array([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dtype=float)

label_issues = []
image_rows = []
expected_image_counts = Counter()
expected_box_counts = Counter()
expected_split_images = {}
dataset_state = hashlib.sha1()
dataset_state.update(sha1_file(SOURCE_DATA_YAML).encode('utf-8'))
dataset_state.update(sha1_file(DATA_YAML).encode('utf-8'))
dataset_state.update(json.dumps(NAMES, separators=(',', ':')).encode('utf-8'))

for split in SPLITS:
    images = split_images(split)
    expected_split_images[split] = len(images)
    label_dir = DATA_ROOT / split / 'labels'
    image_stems = {path.stem for path in images}
    orphan_labels = sorted(path.name for path in label_dir.glob('*.txt') if path.stem not in image_stems)
    assert not orphan_labels, f'Orphan labels in {split}: {orphan_labels[:20]}'

    for image_path in tqdm(images, desc=f'Auditing direct Roboflow {split}'):
        label_path = label_path_for_image(image_path)
        rows, issues = parse_label_file(label_path, split, image_path.name)
        label_issues.extend(issues)
        try:
            with Image.open(image_path) as image:
                image.verify()
        except Exception as exc:
            label_issues.append({
                'split': split, 'file': image_path.name, 'line': 0,
                'reason': 'unreadable_image', 'raw': repr(exc),
            })
        image_sha1 = sha1_file(image_path)
        label_sha1 = sha1_file(label_path) if label_path.exists() else ''
        dataset_state.update(
            f'{split}|{image_path.name}|{image_sha1}|{label_sha1}\n'.encode('utf-8')
        )
        seen_classes = set()
        for class_id, *_ in rows:
            class_name = NAMES[class_id]
            expected_box_counts[(split, class_name)] += 1
            seen_classes.add(class_name)
        for class_name in seen_classes:
            expected_image_counts[(split, class_name)] += 1
        image_rows.append({
            'split': split,
            'image': str(image_path),
            'label': str(label_path),
            'image_sha1': image_sha1,
            'label_sha1': label_sha1,
            'box_count': len(rows),
            'is_background': len(rows) == 0,
        })

issue_df = pd.DataFrame(label_issues)
if len(issue_df):
    issue_df.to_csv(AUDIT_DIR / 'dataset_issues.csv', index=False)
    display(issue_df.head(100))
assert issue_df.empty, f'Dataset audit found {len(issue_df)} structural issues.'

hash_df = pd.DataFrame(image_rows)
hash_df.to_csv(AUDIT_DIR / 'direct_roboflow_image_manifest.csv', index=False)
duplicate_rows = hash_df.groupby('image_sha1').filter(lambda group: len(group) > 1)
duplicate_group_summary = (
    duplicate_rows.groupby('image_sha1')
    .agg(file_count=('image', 'size'), split_count=('split', 'nunique'), splits=('split', lambda x: ','.join(sorted(set(x)))))
    .reset_index()
    if len(duplicate_rows) else pd.DataFrame(columns=['image_sha1', 'file_count', 'split_count', 'splits'])
)
duplicate_group_summary.to_csv(AUDIT_DIR / 'exact_duplicate_groups.csv', index=False)
cross_split_hashes = int((duplicate_group_summary['split_count'] > 1).sum())
assert cross_split_hashes == 0, (
    f'Found {cross_split_hashes} exact image hash groups crossing splits. '
    'Fix the split in Roboflow and create a new dataset version before training.'
)

expected_class_rows = []
for class_id, class_name in enumerate(NAMES):
    row = {'class_id': class_id, 'class': class_name}
    for split in SPLITS:
        row[f'{split}_images'] = expected_image_counts[(split, class_name)]
        row[f'{split}_boxes'] = expected_box_counts[(split, class_name)]
    row['total_images'] = sum(row[f'{split}_images'] for split in SPLITS)
    row['total_boxes'] = sum(row[f'{split}_boxes'] for split in SPLITS)
    expected_class_rows.append(row)

expected_post_dedupe_class_df = pd.DataFrame(expected_class_rows)
expected_post_dedupe_boxes = int(expected_post_dedupe_class_df['total_boxes'].sum())
post_dedupe_state_sha1 = dataset_state.hexdigest()
initial_images = int(len(hash_df))
initial_boxes = expected_post_dedupe_boxes
dedupe_df = pd.DataFrame()
duplicate_label_conflicts = 0
checkpoint_summary = {
    'source': 'roboflow_sdk',
    'workspace': ROBOFLOW_WORKSPACE,
    'project': ROBOFLOW_PROJECT,
    'version': ROBOFLOW_VERSION,
    'format': ROBOFLOW_FORMAT,
    'images': initial_images,
    'boxes': initial_boxes,
    'classes': len(NAMES),
}
expected_post_dedupe_class_df.to_csv(AUDIT_DIR / 'direct_roboflow_class_summary.csv', index=False)
(AUDIT_DIR / 'direct_dataset_state.json').write_text(json.dumps({
    **checkpoint_summary,
    'dataset_state_sha1': post_dedupe_state_sha1,
    'split_images': expected_split_images,
    'exact_duplicate_groups': int(len(duplicate_group_summary)),
    'cross_split_exact_duplicate_groups': cross_split_hashes,
    'image_rewriting': False,
    'image_quality_filtering': False,
}, indent=2))

print('Images:', initial_images)
print('Boxes:', initial_boxes)
print('Background/empty-label images:', int(hash_df['is_background'].sum()))
print('Exact duplicate groups:', len(duplicate_group_summary))
print('Cross-split exact duplicate groups:', cross_split_hashes)
print('Dataset state SHA1:', post_dedupe_state_sha1)

In [ ]:
# Cell 4: Confirm direct-use policy; inspect dimensions and EXIF without rewriting images
direct_rows = []
for row in tqdm(hash_df.to_dict('records'), desc='Inspecting direct Roboflow image metadata'):
    image_path = Path(row['image'])
    with Image.open(image_path) as image:
        width, height = image.size
        exif_orientation = int(image.getexif().get(274, 1) or 1)
    direct_rows.append({
        **row,
        'width': width,
        'height': height,
        'exif_orientation': exif_orientation,
        'orientation_applied': False,
    })

preprocess_df = pd.DataFrame(direct_rows)
nonstandard_orientation = preprocess_df[preprocess_df['exif_orientation'] != 1]
assert nonstandard_orientation.empty, (
    'Roboflow export contains unresolved EXIF orientation metadata. '
    'Because Trial 9 forbids preprocessing, normalize these images in Roboflow first.'
)
assert int(preprocess_df['is_background'].sum()) == 0, (
    'Empty-label images were found but are not verified background RoIs. '
    'Remove or explicitly verify them in Roboflow before this direct-use run.'
)

preprocess_signature = {
    'source': f'roboflow:{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}',
    'format': ROBOFLOW_FORMAT,
    'method': 'direct_roboflow_export_no_image_rewriting',
    'training_imgsz': IMGSZ,
    'runtime_resize': 'ultralytics_letterbox_only',
    'image_rewriting': False,
    'image_quality_filtering': False,
    'auto_orientation': False,
    'expected_images': initial_images,
    'expected_boxes': initial_boxes,
    'expected_split_images': expected_split_images,
    'dataset_state_sha1': post_dedupe_state_sha1,
}
preprocess_df.to_csv(AUDIT_DIR / 'direct_image_metadata.csv', index=False)
(AUDIT_DIR / 'direct_use_policy.json').write_text(json.dumps(preprocess_signature, indent=2))

dimensions = (
    preprocess_df.groupby(['width', 'height']).size()
    .reset_index(name='images')
    .sort_values('images', ascending=False)
)
dimensions.to_csv(AUDIT_DIR / 'source_image_dimensions.csv', index=False)
display(dimensions.head(30))
print('No images or labels were modified.')
print('Ultralytics will letterbox each batch to imgsz=', IMGSZ)

In [ ]:
# Cell 5: Final direct-dataset audit, balance chart, and training gate
def audit_dataset():
    split_rows = []
    class_image_counts = Counter()
    class_box_counts = Counter()
    bad_rows = []
    for split in SPLITS:
        images = split_images(split)
        empty_count = 0
        for image_path in images:
            rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
            bad_rows.extend(issues)
            if not rows:
                empty_count += 1
            present = set()
            for class_id, *_ in rows:
                class_name = NAMES[class_id]
                class_box_counts[(split, class_name)] += 1
                present.add(class_name)
            for class_name in present:
                class_image_counts[(split, class_name)] += 1
        split_rows.append({
            'split': split,
            'images': len(images),
            'empty_label_images': empty_count,
        })

    class_rows = []
    for class_id, class_name in enumerate(NAMES):
        row = {'class_id': class_id, 'class': class_name}
        for split in SPLITS:
            row[f'{split}_images'] = class_image_counts[(split, class_name)]
            row[f'{split}_boxes'] = class_box_counts[(split, class_name)]
        row['total_images'] = sum(row[f'{split}_images'] for split in SPLITS)
        row['total_boxes'] = sum(row[f'{split}_boxes'] for split in SPLITS)
        class_rows.append(row)
    return pd.DataFrame(split_rows), pd.DataFrame(class_rows), pd.DataFrame(bad_rows)

split_summary, class_summary, bad_labels = audit_dataset()
assert bad_labels.empty
assert len(class_summary) == EXPECTED_CLASSES
assert int(split_summary['images'].sum()) == initial_images
assert int(split_summary['empty_label_images'].sum()) == 0
assert int(class_summary['total_boxes'].sum()) == initial_boxes

comparison_columns = [
    'class_id', 'class',
    *[f'{split}_{kind}' for split in SPLITS for kind in ['images', 'boxes']],
    'total_images', 'total_boxes',
]
pd.testing.assert_frame_equal(
    class_summary[comparison_columns].sort_values('class_id').reset_index(drop=True),
    expected_post_dedupe_class_df[comparison_columns].sort_values('class_id').reset_index(drop=True),
)
for split in SPLITS:
    assert (class_summary[f'{split}_images'] > 0).all(), class_summary[class_summary[f'{split}_images'] == 0]
MIN_VALID_IMAGES_PER_CLASS = 10
MIN_TEST_IMAGES_PER_CLASS = 5

assert (class_summary['valid_images'] >= MIN_VALID_IMAGES_PER_CLASS).all(), (
    f'Classes with fewer than {MIN_VALID_IMAGES_PER_CLASS} validation images '
    'make checkpoint and threshold selection unstable:\n'
    + class_summary.loc[
        class_summary['valid_images'] < MIN_VALID_IMAGES_PER_CLASS,
        ['class', 'valid_images'],
    ].to_string(index=False)
)
assert (class_summary['test_images'] >= MIN_TEST_IMAGES_PER_CLASS).all(), (
    f'Classes with fewer than {MIN_TEST_IMAGES_PER_CLASS} test images '
    'cannot support even a limited frozen-test estimate:\n'
    + class_summary.loc[
        class_summary['test_images'] < MIN_TEST_IMAGES_PER_CLASS,
        ['class', 'test_images'],
    ].to_string(index=False)
)

limited_test_support_df = class_summary[
    class_summary['test_images'] < 10
][['class_id', 'class', 'test_images', 'test_boxes']].copy()
limited_test_support_df.to_csv(
    AUDIT_DIR / 'limited_test_support_classes.csv',
    index=False,
)
if len(limited_test_support_df):
    print(
        'WARNING: these classes pass the test gate but have fewer than '
        '10 test images; interpret their per-class test metrics with extra uncertainty.'
    )
    display(limited_test_support_df)

class_mean = float(class_summary['total_images'].mean())
class_std = float(class_summary['total_images'].std(ddof=0))
balance_summary = {
    'minimum_class_images': int(class_summary['total_images'].min()),
    'maximum_class_images': int(class_summary['total_images'].max()),
    'mean_class_images': class_mean,
    'class_image_cv': class_std / class_mean if class_mean else None,
    'max_to_min_ratio': float(class_summary['total_images'].max() / class_summary['total_images'].min()),
    'classes_below_400_images': class_summary.loc[class_summary['total_images'] < 400, 'class'].tolist(),
    'classes_above_600_images': class_summary.loc[class_summary['total_images'] > 600, 'class'].tolist(),
}

split_summary.to_csv(AUDIT_DIR / 'training_split_summary.csv', index=False)
class_summary.to_csv(AUDIT_DIR / 'training_class_summary.csv', index=False)
(AUDIT_DIR / 'balance_summary.json').write_text(json.dumps(balance_summary, indent=2))
display(split_summary)
display(class_summary.sort_values('total_images'))
print(json.dumps(balance_summary, indent=2))

plt.figure(figsize=(12, 12))
plot_df = class_summary.sort_values('train_images')
sns.barplot(data=plot_df, y='class', x='train_images', color='steelblue')
plt.title('Trial 9 direct Roboflow training images by class')
plt.xlabel('Training images containing class')
plt.ylabel('')
plt.tight_layout()
plt.savefig(CHART_DIR / '01_training_images_by_class.png', dpi=180)
plt.show()
print('Direct Roboflow dataset passed the training gate.')

In [ ]:
# Cell 6: Exact GALVAT weighted metric for the Trial 9 ontology
THRESHOLDS = np.array([0.00, 0.05, 0.10, 0.20, 0.50], dtype=float)

GALVAT_RATINGS = {
    'bathroom_sign': {'fn':[8,8,8,8,9], 'fp':[7,8,8,8,9]},
    'canned_good': {'fn':[2,2,2,3,4], 'fp':[2,2,2,3,5]},
    'cellphone': {'fn':[10,10,10,10,10], 'fp':[9,9,9,9,9]},
    'chair': {'fn':[8,8,8,9,9], 'fp':[10,10,10,10,10]},
    'coffee_mug': {'fn':[8,9,9,10,10], 'fp':[5,5,5,5,6]},
    'deodorant': {'fn':[6,6,7,7,8], 'fp':[6,6,7,7,7]},
    'door': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'drinking_glass': {'fn':[8,8,9,9,10], 'fp':[5,5,5,5,6]},
    'electrical_outlet': {'fn':[7,7,8,8,9], 'fp':[7,7,8,8,9]},
    'exit_sign': {'fn':[7,7,7,8,8], 'fp':[7,8,8,8,9]},
    'eyeglasses': {'fn':[8,8,8,9,10], 'fp':[3,3,4,5,6]},
    'fork': {'fn':[7,7,7,8,9], 'fp':[6,7,7,8,8]},
    'hearing_aid': {'fn':[9,9,9,10,10], 'fp':[4,4,5,6,7]},
    'jug': {'fn':[6,6,6,7,8], 'fp':[5,5,6,6,7]},
    'knife': {'fn':[10,10,10,10,10], 'fp':[6,7,7,8,8]},
    'lemon': {'fn':[4,4,5,6,7], 'fp':[4,4,5,6,7]},
    'light_switch': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
    'measuring_cup': {'fn':[4,4,5,6,7], 'fp':[3,3,3,3,4]},
    'paper_towel_dispenser': {'fn':[5,5,6,7,7], 'fp':[5,5,6,7,7]},
    'pot': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
    'razor': {'fn':[8,8,8,9,9], 'fp':[6,7,7,8,8]},
    'salt_pepper_shaker': {'fn':[6,6,6,7,8], 'fp':[2,2,2,3,4]},
    'soap_dispenser': {'fn':[6,6,7,7,8], 'fp':[7,7,7,8,8]},
    'spoon': {'fn':[5,5,5,5,6], 'fp':[5,5,5,5,6]},
    'stairs': {'fn':[10,10,10,10,10], 'fp':[8,8,8,9,10]},
    'table': {'fn':[7,7,7,8,8], 'fp':[7,7,7,8,8]},
    'thermostat': {'fn':[2,2,2,2,3], 'fp':[2,2,2,2,3]},
    'toilet': {'fn':[9,9,9,10,10], 'fp':[10,10,10,10,10]},
    'toothbrush': {'fn':[6,6,7,7,8], 'fp':[7,7,8,8,9]},
    'toothpaste': {'fn':[8,8,9,9,9], 'fp':[8,8,9,9,9]},
    'trash_can': {'fn':[5,5,5,5,6], 'fp':[6,6,7,7,7]},
    'tupperware': {'fn':[3,3,3,4,5], 'fp':[2,3,3,4,5]},
    'urinal': {'fn':[9,9,9,10,10], 'fp':[10,10,10,10,10]},
    'wallet': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'window': {'fn':[4,4,4,5,5], 'fp':[7,8,8,8,8]},
    'wine_glass': {'fn':[8,8,9,9,10], 'fp':[5,5,5,5,6]},
    'writing_device': {'fn':[2,2,3,4,4], 'fp':[1,2,3,3,4]},
}
GALVAT_RATINGS.update({
    'keys': {'fn':[10,10,10,10,10], 'fp':[8,8,8,8,8]},
    'medicine_bottle': {'fn':[10,10,10,10,10], 'fp':[10,10,10,10,10]},
    'cane': {'fn':[10,10,10,10,10], 'fp':[8,8,8,8,8]},
    'tv_remote': {'fn':[4,4,5,6,7], 'fp':[3,3,3,4,5]},
    'purse': {'fn':[9,9,9,9,9], 'fp':[10,10,10,10,10]},
    'railing': {'fn':[8,8,8,9,9], 'fp':[7,7,8,8,9]},
    'pan': {'fn':[7,7,8,8,9], 'fp':[5,5,5,5,6]},
})

def auc(x, y):
    fn = getattr(np, 'trapezoid', np.trapz)
    return float(fn(np.asarray(y, dtype=float), np.asarray(x, dtype=float)))

def remaining_area(error_rate, ratings):
    possible = auc(THRESHOLDS, ratings)
    error = min(max(float(error_rate), 0.0), 1.0)
    if error >= 0.50:
        return 0.0, possible, 0.0
    xs = np.asarray([error, *[t for t in THRESHOLDS if t > error]])
    ys = np.interp(xs, THRESHOLDS, np.asarray(ratings, dtype=float))
    remaining = auc(xs, ys)
    return remaining, possible, remaining / possible if possible else 0.0

def add_galvat_scores(metrics_df):
    missing = sorted(set(metrics_df['class']) - set(GALVAT_RATINGS))
    assert not missing, f'Missing ratings: {missing}'
    output = []
    for row in metrics_df.to_dict('records'):
        precision = min(max(float(row['precision']), 0), 1)
        recall = min(max(float(row['recall']), 0), 1)
        fn_remaining, fn_possible, fn_score = remaining_area(1 - recall, GALVAT_RATINGS[row['class']]['fn'])
        fp_remaining, fp_possible, fp_score = remaining_area(1 - precision, GALVAT_RATINGS[row['class']]['fp'])
        row.update({
            'false_discovery_rate': 1 - precision,
            'false_negative_rate': 1 - recall,
            'fn_score': fn_score,
            'fp_score': fp_score,
            'earned_weighted_points': fn_remaining + fp_remaining,
            'possible_weight': fn_possible + fp_possible,
            'lost_weighted_points': fn_possible + fp_possible - fn_remaining - fp_remaining,
        })
        row['class_score'] = row['earned_weighted_points'] / row['possible_weight']
        output.append(row)
    result = pd.DataFrame(output)
    overall = result['earned_weighted_points'].sum() / result['possible_weight'].sum()
    return float(overall), result

assert set(NAMES) == set(GALVAT_RATINGS), sorted(set(NAMES) - set(GALVAT_RATINGS))
print('Exact GALVAT ratings loaded for all', len(NAMES), 'classes.')

In [ ]:
# Cell 7: Train YOLO26s with periodic Drive checkpoints; safely resume interruptions
TRAIN_DIR = RUN_ROOT / 'train'
LAST_PT = TRAIN_DIR / 'weights' / 'last.pt'
BEST_PT = TRAIN_DIR / 'weights' / 'best.pt'
COMPLETE_MARKER = TRAIN_DIR / 'TRAINING_COMPLETE.json'

train_args = dict(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=32,
    device=0,
    workers=8,
    cache='disk',
    optimizer='AdamW',
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5.0,
    patience=30,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    freeze=0,
    cos_lr=True,
    mosaic=0.25,
    close_mosaic=15,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    erasing=0.0,
    auto_augment=None,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=0.0,
    translate=0.05,
    scale=0.25,
    shear=0.0,
    perspective=0.0,
    save=True,
    save_period=SAVE_PERIOD,
    plots=True,
    amp=True,
    project=str(RUN_ROOT),
    name='train',
    exist_ok=True,
    verbose=True,
)
train_configuration = {
    'model_init': MODEL_INIT,
    'preprocess_signature': preprocess_signature,
    'train_args': {key: str(value) for key, value in train_args.items()},
}
TRAIN_CONFIG_SHA1 = hashlib.sha1(
    json.dumps(train_configuration, sort_keys=True).encode('utf-8')
).hexdigest()
train_configuration['configuration_sha1'] = TRAIN_CONFIG_SHA1
TRAIN_CONFIG_PATH = RUN_ROOT / 'train_configuration.json'
if TRAIN_CONFIG_PATH.exists():
    prior_train_configuration = json.loads(TRAIN_CONFIG_PATH.read_text())
    assert prior_train_configuration.get('configuration_sha1') == TRAIN_CONFIG_SHA1, (
        'The stable Drive run folder contains a different training configuration. '
        'Increment RUN_NAME before starting a different experiment.'
    )
else:
    TRAIN_CONFIG_PATH.write_text(json.dumps(train_configuration, indent=2))

if COMPLETE_MARKER.exists():
    completed_training = json.loads(COMPLETE_MARKER.read_text())
    assert completed_training.get('configuration_sha1') == TRAIN_CONFIG_SHA1
    print('Training already completed:', COMPLETE_MARKER)
elif LAST_PT.exists():
    print('Resuming interrupted run from:', LAST_PT)
    model = YOLO(str(LAST_PT))
    model.train(resume=True)
    COMPLETE_MARKER.write_text(json.dumps({
        'resumed': True,
        'completed_at': str(datetime.datetime.now()),
        'configuration_sha1': TRAIN_CONFIG_SHA1,
    }, indent=2))
else:
    print('Starting:', MODEL_INIT)
    model = YOLO(MODEL_INIT)
    model.train(**train_args)
    COMPLETE_MARKER.write_text(json.dumps({
        'resumed': False,
        'completed_at': str(datetime.datetime.now()),
        'configuration_sha1': TRAIN_CONFIG_SHA1,
    }, indent=2))

assert BEST_PT.exists(), f'Missing: {BEST_PT}'
assert LAST_PT.exists(), f'Missing: {LAST_PT}'
print('BEST_PT:', BEST_PT)
print('LAST_PT:', LAST_PT)

In [ ]:
# Cell 8: Exact fixed-threshold detector evaluation helpers
def load_ground_truth(split):
    cache = {}
    for image_path in split_images(split):
        rows, issues = parse_label_file(label_path_for_image(image_path), split, image_path.name)
        assert not issues
        by_class = defaultdict(list)
        for row in rows:
            by_class[row[0]].append(xywh_to_xyxy(row))
        cache[image_path.name] = {
            class_id: np.asarray(boxes, dtype=float)
            for class_id, boxes in by_class.items()
        }
    return cache

GT_CACHE = {split: load_ground_truth(split) for split in ['valid', 'test']}

def collect_predictions(model_path, split, nms_iou, source_paths=None):
    paths = list(source_paths) if source_paths is not None else split_images(split)
    assert paths, f'No prediction images for split: {split}'
    path_keys = [Path(path).name for path in paths]
    assert len(set(path_keys)) == len(path_keys), (
        f'Duplicate image basenames in {split}; filename-keyed evaluation is ambiguous'
    )
    predictions = {}
    detector = None
    progress = tqdm(total=len(paths), desc=f'Predict {split} iou={nms_iou}')
    try:
        detector = YOLO(str(model_path))
        for start in range(0, len(paths), PRED_BATCH):
            chunk = paths[start:start + PRED_BATCH]
            chunk_results = detector.predict(
                source=[str(path) for path in chunk],
                stream=False,
                imgsz=IMGSZ,
                conf=PRED_MIN_CONF,
                iou=float(nms_iou),
                max_det=MAX_DET,
                device=0,
                batch=len(chunk),
                rect=False,
                agnostic_nms=False,
                verbose=False,
            )
            assert len(chunk_results) == len(chunk)
            # Ultralytics may retain stale result.path metadata when the same
            # predictor receives repeated path-list calls. Results preserve
            # input order, so key each result from its paired source path.
            for source_path, result in zip(chunk, chunk_results):
                key = Path(source_path).name
                assert key not in predictions, f'Duplicate prediction key: {key}'
                if result.boxes is None or len(result.boxes) == 0:
                    predictions[key] = {
                        'class_id': np.empty(0, dtype=int),
                        'confidence': np.empty(0, dtype=float),
                        'boxes': np.empty((0, 4), dtype=float),
                    }
                else:
                    predictions[key] = {
                        'class_id': result.boxes.cls.detach().cpu().numpy().astype(int),
                        'confidence': result.boxes.conf.detach().cpu().numpy().astype(float),
                        'boxes': result.boxes.xyxyn.detach().cpu().numpy().astype(float),
                    }
            del chunk_results
            progress.update(len(chunk))
    finally:
        progress.close()
        if detector is not None and getattr(detector, 'model', None) is not None:
            detector.model.to('cpu')
        del detector
        gc.collect()
        torch.cuda.empty_cache()
    assert len(predictions) == len(paths), (
        f'Prediction key collision or missing result: {len(predictions)} / {len(paths)}'
    )
    return predictions

def pairwise_iou(pred_box, gt_boxes):
    if len(gt_boxes) == 0:
        return np.empty(0, dtype=float)
    top_left = np.maximum(pred_box[:2], gt_boxes[:, :2])
    bottom_right = np.minimum(pred_box[2:], gt_boxes[:, 2:])
    wh = np.maximum(0.0, bottom_right - top_left)
    inter = wh[:, 0] * wh[:, 1]
    pred_area = max(0.0, pred_box[2] - pred_box[0]) * max(0.0, pred_box[3] - pred_box[1])
    gt_area = np.maximum(0.0, gt_boxes[:, 2] - gt_boxes[:, 0]) * np.maximum(0.0, gt_boxes[:, 3] - gt_boxes[:, 1])
    union = pred_area + gt_area - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

def match_one_class(gt_boxes, pred_boxes, pred_conf, threshold):
    keep = pred_conf >= float(threshold)
    pred_boxes = pred_boxes[keep]
    pred_conf = pred_conf[keep]
    order = np.argsort(-pred_conf)
    matched = np.zeros(len(gt_boxes), dtype=bool)
    tp = fp = 0
    for pred_box in pred_boxes[order]:
        ious = pairwise_iou(pred_box, gt_boxes)
        if len(ious):
            ious[matched] = -1
            best = int(np.argmax(ious))
            if ious[best] >= MATCH_IOU:
                matched[best] = True
                tp += 1
                continue
        fp += 1
    return tp, fp, int((~matched).sum())

def evaluate_thresholds(gt_cache, prediction_cache, confidence_values):
    rows = []
    for threshold in confidence_values:
        counts = {class_id: [0, 0, 0] for class_id in range(len(NAMES))}
        for image_name, gt_by_class in gt_cache.items():
            pred = prediction_cache.get(image_name, {
                'class_id': np.empty(0, dtype=int),
                'confidence': np.empty(0),
                'boxes': np.empty((0, 4)),
            })
            active = set(gt_by_class) | set(pred['class_id'].tolist())
            for class_id in active:
                gt_boxes = gt_by_class.get(class_id, np.empty((0, 4)))
                mask = pred['class_id'] == class_id
                tp, fp, fn = match_one_class(
                    gt_boxes, pred['boxes'][mask], pred['confidence'][mask], threshold
                )
                counts[class_id][0] += tp
                counts[class_id][1] += fp
                counts[class_id][2] += fn
        metric_rows = []
        for class_id, class_name in enumerate(NAMES):
            tp, fp, fn = counts[class_id]
            metric_rows.append({
                'class_id': class_id,
                'class': class_name,
                'confidence_threshold': float(threshold),
                'tp': tp, 'fp': fp, 'fn': fn,
                'precision': tp / (tp + fp) if tp + fp else 0.0,
                'recall': tp / (tp + fn) if tp + fn else 0.0,
            })
        score, scored = add_galvat_scores(pd.DataFrame(metric_rows))
        scored['overall_global_galvat'] = score
        rows.append(scored)
    return pd.concat(rows, ignore_index=True)

def select_per_class_thresholds(all_threshold_rows):
    selected = []
    for class_name, group in all_threshold_rows.groupby('class'):
        best = group.sort_values(
            ['class_score', 'recall', 'precision', 'confidence_threshold'],
            ascending=[False, False, False, False],
        ).iloc[0]
        selected.append(best)
    selected_df = pd.DataFrame(selected).sort_values('class_id')
    overall = selected_df['earned_weighted_points'].sum() / selected_df['possible_weight'].sum()
    return float(overall), selected_df

def evaluate_threshold_map(gt_cache, prediction_cache, thresholds_by_class):
    thresholds = sorted(set(float(v) for v in thresholds_by_class.values()))
    all_rows = evaluate_thresholds(gt_cache, prediction_cache, thresholds)
    selected = []
    for class_name in NAMES:
        threshold = float(thresholds_by_class[class_name])
        row = all_rows[(all_rows['class'] == class_name) & np.isclose(all_rows['confidence_threshold'], threshold)].iloc[0]
        selected.append(row)
    result = pd.DataFrame(selected).sort_values('class_id')
    overall = result['earned_weighted_points'].sum() / result['possible_weight'].sum()
    return float(overall), result

In [ ]:
# Cell 9: Restart-safe validation checkpoint, NMS, and class-threshold selection
# A completed training call can leave its trainer/model attached to the notebook.
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
print(
    'CUDA before selection (allocated/reserved GB):',
    round(torch.cuda.memory_allocated() / 1e9, 3),
    round(torch.cuda.memory_reserved() / 1e9, 3),
)

def epoch_number(path):
    digits = ''.join(ch for ch in path.stem if ch.isdigit())
    return int(digits) if digits else 10**9

periodic = sorted((TRAIN_DIR / 'weights').glob('epoch*.pt'), key=epoch_number)
checkpoint_candidates = []
checkpoint_sha1_by_path = {}
seen_checkpoint_hashes = set()
for path in [*periodic, BEST_PT, LAST_PT]:
    if not path.exists():
        continue
    checkpoint_sha1 = sha1_file(path)
    if checkpoint_sha1 in seen_checkpoint_hashes:
        print('Skipping byte-identical checkpoint:', path.name)
        continue
    seen_checkpoint_hashes.add(checkpoint_sha1)
    checkpoint_candidates.append(path)
    checkpoint_sha1_by_path[str(path)] = checkpoint_sha1
assert checkpoint_candidates
print('Checkpoint candidates:', [p.name for p in checkpoint_candidates])

global_rows = []
class_candidate_rows = []
leaderboard_rows = []
PAIR_CACHE_DIR = SELECTION_DIR / 'pair_cache'
PAIR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for checkpoint in checkpoint_candidates:
    checkpoint_sha1 = checkpoint_sha1_by_path[str(checkpoint)]
    for nms_iou in NMS_IOU_VALUES:
        pair_name = (
            f'{checkpoint.stem}_{checkpoint_sha1[:12]}_'
            f'nms_{str(nms_iou).replace(".", "p")}'
        )
        pair_dir = PAIR_CACHE_DIR / pair_name
        pair_dir.mkdir(parents=True, exist_ok=True)
        pair_csv = pair_dir / 'validation_per_class_thresholds.csv'
        pair_done = pair_dir / 'COMPLETE.json'

        if pair_csv.exists() and pair_done.exists():
            threshold_rows = pd.read_csv(pair_csv)
            print('Loaded cached validation pair:', pair_name)
        else:
            predictions = collect_predictions(checkpoint, 'valid', nms_iou)
            threshold_rows = evaluate_thresholds(GT_CACHE['valid'], predictions, CONF_VALUES)
            threshold_rows['checkpoint'] = str(checkpoint)
            threshold_rows['checkpoint_name'] = checkpoint.name
            threshold_rows['checkpoint_sha1'] = checkpoint_sha1
            threshold_rows['nms_iou'] = nms_iou
            temporary_csv = pair_csv.with_suffix('.csv.part')
            threshold_rows.to_csv(temporary_csv, index=False)
            temporary_csv.replace(pair_csv)
            pair_done.write_text(json.dumps({
                'checkpoint': str(checkpoint),
                'checkpoint_sha1': checkpoint_sha1,
                'nms_iou': nms_iou,
                'completed_at': str(datetime.datetime.now()),
            }, indent=2))
            del predictions
            torch.cuda.empty_cache()

        assert len(threshold_rows) == len(NAMES) * len(CONF_VALUES)
        class_candidate_rows.append(threshold_rows)

        global_summary = (
            threshold_rows.groupby('confidence_threshold', as_index=False)
            .agg(
                galvat=('overall_global_galvat', 'first'),
                macro_precision=('precision', 'mean'),
                macro_recall=('recall', 'mean'),
                total_tp=('tp', 'sum'), total_fp=('fp', 'sum'), total_fn=('fn', 'sum'),
            )
        )
        global_summary['checkpoint'] = str(checkpoint)
        global_summary['checkpoint_name'] = checkpoint.name
        global_summary['nms_iou'] = nms_iou
        global_rows.append(global_summary)

        classwise_score, classwise = select_per_class_thresholds(threshold_rows)
        best_global = global_summary.sort_values(
            ['galvat', 'macro_recall', 'macro_precision'], ascending=False
        ).iloc[0]
        leaderboard_rows.append({
            'checkpoint': str(checkpoint),
            'checkpoint_name': checkpoint.name,
            'nms_iou': nms_iou,
            'per_class_threshold_galvat': classwise_score,
            'per_class_macro_precision': float(classwise['precision'].mean()),
            'per_class_macro_recall': float(classwise['recall'].mean()),
            'best_global_threshold': float(best_global['confidence_threshold']),
            'best_global_galvat': float(best_global['galvat']),
        })
        del threshold_rows
        gc.collect()

validation_global_df = pd.concat(global_rows, ignore_index=True)
validation_classes_df = pd.concat(class_candidate_rows, ignore_index=True)
leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values(
    ['per_class_threshold_galvat', 'per_class_macro_recall', 'best_global_galvat'],
    ascending=False,
)
validation_global_df.to_csv(SELECTION_DIR / 'validation_global_threshold_sweep.csv', index=False)
validation_classes_df.to_csv(SELECTION_DIR / 'validation_per_class_threshold_candidates.csv', index=False)
leaderboard_df.to_csv(SELECTION_DIR / 'validation_checkpoint_nms_leaderboard.csv', index=False)
display(leaderboard_df)

winner = leaderboard_df.iloc[0]
SELECTED_PT = Path(winner['checkpoint'])
SELECTED_NMS_IOU = float(winner['nms_iou'])
SELECTED_GLOBAL_CONF = float(winner['best_global_threshold'])
selected_candidates = validation_classes_df[
    (validation_classes_df['checkpoint'] == str(SELECTED_PT))
    & np.isclose(validation_classes_df['nms_iou'], SELECTED_NMS_IOU)
]
SELECTED_VAL_GALVAT, selected_threshold_rows = select_per_class_thresholds(selected_candidates)
CLASS_THRESHOLDS = {
    row['class']: float(row['confidence_threshold'])
    for row in selected_threshold_rows.to_dict('records')
}
SELECTED_PT_SHA1 = sha1_file(SELECTED_PT)
THRESHOLD_POLICY_SHA1 = hashlib.sha1(
    json.dumps(CLASS_THRESHOLDS, sort_keys=True).encode('utf-8')
).hexdigest()
selected_threshold_rows.to_csv(SELECTION_DIR / 'selected_per_class_thresholds_and_val_metrics.csv', index=False)
(SELECTION_DIR / 'selected_per_class_thresholds.json').write_text(json.dumps(CLASS_THRESHOLDS, indent=2))

selection = {
    'checkpoint': str(SELECTED_PT),
    'checkpoint_name': SELECTED_PT.name,
    'checkpoint_sha1': SELECTED_PT_SHA1,
    'nms_iou': SELECTED_NMS_IOU,
    'best_global_conf': SELECTED_GLOBAL_CONF,
    'best_global_val_galvat': float(winner['best_global_galvat']),
    'per_class_threshold_val_galvat': SELECTED_VAL_GALVAT,
    'match_iou': MATCH_IOU,
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'selected_on': 'validation_only',
}
(SELECTION_DIR / 'selection.json').write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

In [ ]:
# Cell 10: Final validation report with exact selected thresholds and official AP
def official_ap_dataframe(metrics):
    summary = pd.DataFrame(metrics.summary())
    if summary.empty:
        return pd.DataFrame({'class': NAMES, 'ap50': 0.0, 'map50_95': 0.0})
    renamed = summary.rename(columns={
        'Class': 'class', 'mAP50': 'ap50', 'mAP50-95': 'map50_95',
        'Images': 'official_images', 'Instances': 'official_instances',
    })
    return renamed[['class', 'ap50', 'map50_95', 'official_images', 'official_instances']]

selected_model = YOLO(str(SELECTED_PT))
official_val = selected_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMGSZ, conf=0.001,
    iou=SELECTED_NMS_IOU, device=0, plots=True, save_json=False,
    project=str(SELECTION_DIR), name='official_val_ap', exist_ok=True, verbose=False,
)
val_predictions = collect_predictions(SELECTED_PT, 'valid', SELECTED_NMS_IOU)
val_galvat, val_exact = evaluate_threshold_map(GT_CACHE['valid'], val_predictions, CLASS_THRESHOLDS)
val_report = val_exact.merge(official_ap_dataframe(official_val), on='class', how='left')
val_report = val_report.merge(
    class_summary[['class', 'valid_images', 'valid_boxes']], on='class', how='left'
).sort_values('class_score')
val_report[['ap50', 'map50_95']] = val_report[['ap50', 'map50_95']].fillna(0.0)
val_report.to_csv(SELECTION_DIR / 'validation_selected_per_class_metrics.csv', index=False)
assert len(val_report) == 44
assert abs(val_galvat - SELECTED_VAL_GALVAT) < 1e-12
print('Frozen validation GALVAT:', val_galvat)
display(val_report)

In [ ]:
# Cell 11: One-shot test evaluation using frozen validation settings
TEST_LOCK = TEST_DIR / 'TEST_EVALUATION_COMPLETE.json'
frozen_test_signature = {
    'checkpoint_sha1': SELECTED_PT_SHA1,
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'nms_iou': SELECTED_NMS_IOU,
    'match_iou': MATCH_IOU,
    'preprocess_source_state_sha1': post_dedupe_state_sha1,
}
if TEST_LOCK.exists():
    prior_test_lock = json.loads(TEST_LOCK.read_text())
    assert all(
        prior_test_lock.get(key) == value
        for key, value in frozen_test_signature.items()
    ), (
        'The frozen test lock belongs to different model/data/threshold settings. '
        'Increment RUN_NAME; do not overwrite or retune an existing test evaluation.'
    )
    test_report = pd.read_csv(TEST_DIR / 'test_frozen_per_class_metrics.csv')
    test_summary = json.loads((TEST_DIR / 'test_frozen_summary.json').read_text())
    TEST_GALVAT = float(test_summary['galvat_weighted_score'])
    print('Loaded completed frozen test evaluation.')
else:
    test_predictions = collect_predictions(SELECTED_PT, 'test', SELECTED_NMS_IOU)
    TEST_GALVAT, test_exact = evaluate_threshold_map(GT_CACHE['test'], test_predictions, CLASS_THRESHOLDS)
    official_test = selected_model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, conf=0.001,
        iou=SELECTED_NMS_IOU, device=0, plots=True, save_json=True,
        project=str(TEST_DIR), name='official_test_ap', exist_ok=True, verbose=False,
    )
    test_report = test_exact.merge(official_ap_dataframe(official_test), on='class', how='left')
    test_report = test_report.merge(
        class_summary[['class', 'test_images', 'test_boxes']], on='class', how='left'
    ).sort_values('class_score')
    test_report[['ap50', 'map50_95']] = test_report[['ap50', 'map50_95']].fillna(0.0)
    test_report.to_csv(TEST_DIR / 'test_frozen_per_class_metrics.csv', index=False)
    test_summary = {
        'galvat_weighted_score': TEST_GALVAT,
        'macro_precision': float(test_report['precision'].mean()),
        'macro_recall': float(test_report['recall'].mean()),
        'macro_ap50': float(test_report['ap50'].mean()),
        'macro_map50_95': float(test_report['map50_95'].mean()),
        'classes': len(test_report),
        'classes_below_0_2': int((test_report['class_score'] < 0.2).sum()),
        'zero_score_classes': int((test_report['class_score'] <= 0).sum()),
        'checkpoint': str(SELECTED_PT),
        'nms_iou': SELECTED_NMS_IOU,
        'threshold_policy': 'validation_selected_per_class',
        'test_tuning_performed': False,
    }
    (TEST_DIR / 'test_frozen_summary.json').write_text(json.dumps(test_summary, indent=2))
    TEST_LOCK.write_text(json.dumps({
        **frozen_test_signature,
        'completed_at': str(datetime.datetime.now()),
    }, indent=2))

print(json.dumps(test_summary, indent=2))
print('Weakest classes:')
display(test_report.head(15))

In [ ]:
# Cell 12: Frozen-threshold spoken burden and fresh background-RoI evaluation
def accepted_prediction_counts(prediction_cache, thresholds):
    rows = []
    for image_name, pred in prediction_cache.items():
        accepted = []
        for class_id, confidence in zip(pred['class_id'], pred['confidence']):
            class_name = NAMES[int(class_id)]
            if confidence >= thresholds[class_name]:
                accepted.append((class_name, float(confidence)))
        rows.append({
            'image': image_name,
            'accepted_detections': len(accepted),
            'classes': ','.join(x[0] for x in accepted),
        })
    return pd.DataFrame(rows)

if 'test_predictions' not in globals():
    test_predictions = collect_predictions(SELECTED_PT, 'test', SELECTED_NMS_IOU)
spoken_df = accepted_prediction_counts(test_predictions, CLASS_THRESHOLDS)
spoken_df.to_csv(TEST_DIR / 'test_spoken_detection_counts.csv', index=False)
SPOKEN_BUDGET = 3

BG_DIR = RUN_ROOT / 'fresh_background_roi'
BG_IMAGES = BG_DIR / 'images'
BG_IMAGES.mkdir(parents=True, exist_ok=True)

def create_background_crops(target=400, crop_size=320, seed=91):
    rng = random.Random(seed)
    candidates = split_images('test')
    rng.shuffle(candidates)
    rows = []
    for image_path in tqdm(candidates, desc='Creating non-overlap background crops'):
        if len(rows) >= target:
            break
        gt_rows, _ = parse_label_file(label_path_for_image(image_path))
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            continue
        width, height = image.size
        size = min(crop_size, width, height)
        if size < 128:
            continue
        gt_abs = []
        for row in gt_rows:
            box = xywh_to_xyxy(row)
            gt_abs.append(np.array([box[0]*width, box[1]*height, box[2]*width, box[3]*height]))
        for _ in range(30):
            x1 = rng.randint(0, width - size)
            y1 = rng.randint(0, height - size)
            crop_box = np.array([x1, y1, x1 + size, y1 + size])
            def intersects(a, b):
                return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])
            if any(intersects(crop_box, gt) for gt in gt_abs):
                continue
            out_path = BG_IMAGES / f'fresh_bg_{len(rows):04d}_{image_path.stem[:60]}.jpg'
            image.crop(tuple(crop_box)).resize((IMGSZ, IMGSZ), Image.Resampling.LANCZOS).save(out_path, quality=92)
            rows.append({'crop': str(out_path), 'source': str(image_path), 'x1': x1, 'y1': y1, 'size': size})
            break
    return pd.DataFrame(rows)

bg_manifest = create_background_crops()
assert len(bg_manifest) >= 100, f'Only created {len(bg_manifest)} background crops.'
bg_manifest.to_csv(BG_DIR / 'background_crop_manifest.csv', index=False)
bg_paths = [Path(path) for path in bg_manifest['crop']]
bg_predictions = collect_predictions(SELECTED_PT, 'fresh_background', SELECTED_NMS_IOU, bg_paths)
bg_counts = accepted_prediction_counts(bg_predictions, CLASS_THRESHOLDS)
bg_counts.to_csv(BG_DIR / 'background_predictions.csv', index=False)

roi_summary = {
    'fresh_background_crop_count': len(bg_counts),
    'fresh_background_any_detection_rate': float((bg_counts['accepted_detections'] > 0).mean()),
    'fresh_background_mean_detections': float(bg_counts['accepted_detections'].mean()),
    'test_spoken_budget': SPOKEN_BUDGET,
    'test_over_budget_rate': float((spoken_df['accepted_detections'] > SPOKEN_BUDGET).mean()),
    'test_mean_accepted_detections': float(spoken_df['accepted_detections'].mean()),
    'settings_frozen_from_validation': True,
}
(RUN_ROOT / 'roi_spoken_summary.json').write_text(json.dumps(roi_summary, indent=2))
print(json.dumps(roi_summary, indent=2))

In [ ]:
# Cell 13: Decision-oriented charts and complete per-class tables
plt.figure(figsize=(10, 6))
sns.barplot(data=leaderboard_df.head(15), y='checkpoint_name', x='per_class_threshold_galvat', hue='nms_iou', dodge=False)
plt.title('Validation GALVAT checkpoint and NMS ranking')
plt.tight_layout()
plt.savefig(CHART_DIR / '02_validation_checkpoint_ranking.png', dpi=180)
plt.show()

plot_test = test_report.sort_values('class_score')
plt.figure(figsize=(11, 12))
sns.barplot(data=plot_test, y='class', x='class_score', color='seagreen')
plt.axvline(0.5, color='firebrick', linestyle='--')
plt.title(f'Trial 9 frozen test GALVAT by class (overall={TEST_GALVAT:.3f})')
plt.tight_layout()
plt.savefig(CHART_DIR / '03_test_galvat_by_class.png', dpi=180)
plt.show()

plt.figure(figsize=(11, 12))
sns.barplot(data=test_report.sort_values('lost_weighted_points', ascending=False), y='class', x='lost_weighted_points', color='firebrick')
plt.title('Test lost GALVAT weighted points by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '04_test_lost_weighted_points.png', dpi=180)
plt.show()

threshold_plot = selected_threshold_rows.sort_values('confidence_threshold')
plt.figure(figsize=(11, 12))
sns.barplot(data=threshold_plot, y='class', x='confidence_threshold', color='darkorange')
plt.xlim(0, 1)
plt.title('Validation-selected confidence threshold by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '05_selected_class_thresholds.png', dpi=180)
plt.show()

plt.figure(figsize=(8, 7))
sns.scatterplot(data=test_report, x='recall', y='precision', size='possible_weight', hue='class_score', palette='viridis', sizes=(40, 250))
plt.title('Frozen test precision-recall by class')
plt.tight_layout()
plt.savefig(CHART_DIR / '06_test_precision_recall.png', dpi=180)
plt.show()

comparison_summary = pd.DataFrame([
    {'run': 'Trial 8 frozen test baseline', 'galvat': 0.7302593034096614},
    {'run': 'Trial 9 validation class thresholds', 'galvat': SELECTED_VAL_GALVAT},
    {'run': 'Trial 9 frozen test', 'galvat': TEST_GALVAT},
])
comparison_summary['delta_vs_trial8_test'] = (
    comparison_summary['galvat'] - 0.7302593034096614
)
comparison_summary.to_csv(RUN_ROOT / 'trial8_trial9_galvat_comparison.csv', index=False)
display(comparison_summary)

In [ ]:
# Cell 14: Copy the selected checkpoint, export ONNX, and smoke-test inference
SELECTED_COPY = EXPORT_DIR / 'galvat_trial9_selected.pt'
shutil.copy2(SELECTED_PT, SELECTED_COPY)
shutil.copy2(SELECTION_DIR / 'selected_per_class_thresholds.json', EXPORT_DIR / 'selected_per_class_thresholds.json')

export_model = YOLO(str(SELECTED_COPY))
exported = Path(export_model.export(
    format='onnx', imgsz=IMGSZ, opset=12, simplify=True,
    dynamic=False, nms=False, batch=1, device=0,
))
ONNX_PATH = EXPORT_DIR / 'galvat_trial9_selected.onnx'
if exported.resolve() != ONNX_PATH.resolve():
    shutil.copy2(exported, ONNX_PATH)
assert ONNX_PATH.exists() and ONNX_PATH.stat().st_size > 0

smoke_image = split_images('test')[0]
onnx_model = YOLO(str(ONNX_PATH), task='detect')
smoke_results = onnx_model.predict(source=str(smoke_image), imgsz=IMGSZ, conf=PRED_MIN_CONF, verbose=False)
assert len(smoke_results) == 1
export_summary = {
    'pytorch': str(SELECTED_COPY),
    'pytorch_sha1': sha1_file(SELECTED_COPY),
    'onnx': str(ONNX_PATH),
    'onnx_sha1': sha1_file(ONNX_PATH),
    'onnx_size_mb': ONNX_PATH.stat().st_size / 1e6,
    'imgsz': IMGSZ,
    'opset': 12,
    'smoke_test_image': str(smoke_image),
    'smoke_test_passed': True,
}
(EXPORT_DIR / 'export_summary.json').write_text(json.dumps(export_summary, indent=2))
print(json.dumps(export_summary, indent=2))

In [ ]:
# Cell 15: Final reproducible Trial 9 summary and compact metrics archive
final_summary = {
    'run_name': RUN_NAME,
    'run_id': 'galvat_trial9_yolo26s_640_balanced_roboflow_v3_planned_v1',
    'created_at': RUN_STAMP,
    'dataset_manifest_id': 'galvat_trial9_roboflow_v3_balanced_candidate_20260808_v1',
    'experiment_plan_id': 'galvat_trial9_balanced_roboflow_train_eval_v1',
    'roboflow_source': checkpoint_summary,
    'dataset_state_sha1': post_dedupe_state_sha1,
    'images': int(split_summary['images'].sum()),
    'boxes': int(class_summary['total_boxes'].sum()),
    'split_images': expected_split_images,
    'class_count': len(NAMES),
    'exact_duplicate_groups': int(len(duplicate_group_summary)),
    'cross_split_exact_duplicate_groups': cross_split_hashes,
    'input_policy': preprocess_signature,
    'model_init': MODEL_INIT,
    'imgsz': IMGSZ,
    'training_configuration_sha1': TRAIN_CONFIG_SHA1,
    'selected_checkpoint': str(SELECTED_PT),
    'selected_checkpoint_sha1': SELECTED_PT_SHA1,
    'selected_model_copy': str(SELECTED_COPY),
    'selected_onnx': str(ONNX_PATH),
    'selected_nms_iou': SELECTED_NMS_IOU,
    'selected_global_conf_comparator': SELECTED_GLOBAL_CONF,
    'threshold_policy': 'validation_selected_per_class',
    'threshold_policy_sha1': THRESHOLD_POLICY_SHA1,
    'validation_galvat': SELECTED_VAL_GALVAT,
    'test_galvat': TEST_GALVAT,
    'test_macro_precision': test_summary['macro_precision'],
    'test_macro_recall': test_summary['macro_recall'],
    'test_macro_ap50': test_summary['macro_ap50'],
    'test_macro_map50_95': test_summary['macro_map50_95'],
    'fresh_background_any_detection_rate': roi_summary['fresh_background_any_detection_rate'],
    'test_spoken_over_budget_rate': roi_summary['test_over_budget_rate'],
    'test_tuning_performed': False,
    'primary_metric': 'full-ontology GALVAT weighted score',
    'trial8_test_galvat_baseline': 0.7302593034096614,
    'promotion_target_test_galvat': 0.75,
    'remaining_validity_risk': (
        'Exact cross-split image hashes are excluded by a hard gate. Near-duplicate and '
        'source/scene leakage remain possible without complete source_id and scene_id metadata. '
        'Balanced object-centric web imagery may underrepresent real clutter and background RoIs.'
    ),
    'deployment_target': 'Magic Leap; YOLO26s retained for the size budget',
}
(RUN_ROOT / 'trial9_final_run_summary.json').write_text(json.dumps(final_summary, indent=2))

METRICS_ZIP = RUN_ROOT / 'trial9_metrics_and_reports.zip'
with zipfile.ZipFile(METRICS_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob('*'):
        if path.is_file() and path != METRICS_ZIP and path.suffix.lower() in {'.csv', '.json', '.png', '.yaml'}:
            archive.write(path, path.relative_to(RUN_ROOT))
with zipfile.ZipFile(METRICS_ZIP) as archive:
    assert archive.testzip() is None

print(json.dumps(final_summary, indent=2))
print('\nFinal Drive run folder:')
print(RUN_ROOT)
print('\nCells are restart-safe. Rerun all cells to resume the same Drive run.')
print('Do not change thresholds after viewing test results.')